# 📓 Notebook 1 — Setup, Data Loading & Exploratory Analysis
**Uncertainty Quantification in ML Classifiers: A Calibration Benchmarking Study**

---
**Works on:** Google Colab · Jupyter Notebook · JupyterLab · VS Code

Run cells top-to-bottom. This notebook loads three scikit-learn datasets,
performs EDA, builds stratified splits, and saves `data/splits.pkl` for the
downstream notebooks. **No external data downloads needed.**


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, sys, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
warnings.filterwarnings('ignore')

from sklearn import datasets as sk_datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
})
PALETTE = {
    'Logistic Regression': '#1d9e75', 'Random Forest':     '#a32d2d',
    'Gradient Boosting':   '#ba7517', 'SVM (RBF)':         '#185fa5',
    'MLP':                 '#533ab7',
}

# ── Path resolution (local Jupyter / VS Code / Google Colab / nbconvert) ─────
DATA_DIR = os.path.join(os.getcwd(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

def data(filename):
    """Always returns an absolute path to data/<filename> in the working directory."""
    return os.path.join(DATA_DIR, filename)

print(f"Working directory : {os.getcwd()}")
print(f"Data directory    : {DATA_DIR}")
print(f"Python            : {sys.version.split()[0]}")


In [ ]:
# ── Dataset loading ───────────────────────────────────────────────────────────
def load_datasets():
    loaders = {
        'breast_cancer': sk_datasets.load_breast_cancer,
        'wine':          sk_datasets.load_wine,
        'diabetes':      sk_datasets.load_diabetes,
    }
    out = {}
    for name, fn in loaders.items():
        raw = fn()
        X = pd.DataFrame(raw.data, columns=raw.feature_names)
        if name == 'diabetes':
            y = (raw.target > np.median(raw.target)).astype(int)
            tnames = ['low-progression', 'high-progression']
        else:
            y, tnames = raw.target, list(raw.target_names)
        out[name] = {'X': X, 'y': y, 'target_names': tnames}
    return out

# ── Split helper ──────────────────────────────────────────────────────────────
def make_splits(X, y, seed=RANDOM_SEED):
    """60% train | 20% calib | 20% test, stratified."""
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.40, random_state=seed, stratify=y)
    X_cal, X_te, y_cal, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp)
    return X_tr, X_cal, X_te, y_tr, y_cal, y_te

def build_and_save_splits():
    """Build splits from scratch and save to data/. Returns splits dict."""
    print("Building splits from scratch...")
    ds = load_datasets()
    splits = {}
    for name, d in ds.items():
        X, y = d['X'].values, d['y']
        X_tr, X_cal, X_te, y_tr, y_cal, y_te = make_splits(X, y)
        sc = StandardScaler().fit(X_tr)
        splits[name] = {
            'X_train': sc.transform(X_tr), 'y_train': y_tr,
            'X_calib': sc.transform(X_cal), 'y_calib': y_cal,
            'X_test':  sc.transform(X_te),  'y_test':  y_te,
        }
    with open(data('splits.pkl'), 'wb') as f: pickle.dump(splits, f)
    with open(data('meta.pkl'),   'wb') as f:
        pickle.dump({k: {'target_names': v['target_names']} for k,v in ds.items()}, f)
    print(f"  Saved {data('splits.pkl')}")
    return splits

def load_splits():
    """Load splits, auto-building them if missing."""
    p = data('splits.pkl')
    if not os.path.exists(p):
        print(f"splits.pkl not found at {p}")
        print("Run Notebook 1 first, or auto-generating now...")
        return build_and_save_splits()
    with open(p, 'rb') as f:
        return pickle.load(f)

# ── Classifiers ───────────────────────────────────────────────────────────────
def get_classifiers():
    return {
        'Logistic Regression': LogisticRegression(
            C=1.0, max_iter=1000, random_state=RANDOM_SEED),
        'Random Forest':       RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
        'Gradient Boosting':   GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED),
        'SVM (RBF)':           SVC(
            kernel='rbf', C=1.0, gamma='scale',
            probability=True, random_state=RANDOM_SEED),
        'MLP':                 MLPClassifier(
            hidden_layer_sizes=(100, 50), activation='relu',
            max_iter=500, random_state=RANDOM_SEED),
    }

# ── Calibration metrics ───────────────────────────────────────────────────────
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        ece += m.sum() / n * abs(float(y_true[m].mean()) - float(y_prob[m].mean()))
    return ece

def compute_mce(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); mce = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        mce = max(mce, abs(float(y_true[m].mean()) - float(y_prob[m].mean())))
    return mce

def brier_score(y_true, y_prob):
    return float(np.mean((np.array(y_true) - np.array(y_prob)) ** 2))

def reliability_data(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    centers, confs, freqs, sizes = [], [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any(): continue
        centers.append((lo+hi)/2); confs.append(float(y_prob[m].mean()))
        freqs.append(float(y_true[m].mean())); sizes.append(int(m.sum()))
    return np.array(centers), np.array(confs), np.array(freqs), np.array(sizes)

def get_probs(clf, X, y):
    """Binary → prob of positive class; multi-class → top-label ECE."""
    proba = clf.predict_proba(X)
    if proba.shape[1] == 2:
        return np.array(y), proba[:, 1]
    return (clf.predict(X) == np.array(y)).astype(int), proba.max(axis=1)

# ── Calibrators ───────────────────────────────────────────────────────────────
class PlattScaler:
    def __init__(self):
        self.lr = LogisticRegression(C=1.0, solver='lbfgs')
        self._fallback = False
    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True
        else:
            self._fallback = False
            self.lr.fit(p.reshape(-1, 1), y)
        return self
    def predict_proba(self, p):
        return p if self._fallback else self.lr.predict_proba(p.reshape(-1,1))[:,1]

class IsotonicScaler:
    def __init__(self):
        self.ir = IsotonicRegression(out_of_bounds='clip')
        self._fallback = False
    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True
        else:
            self._fallback = False
            self.ir.fit(p, y)
        return self
    def predict_proba(self, p):
        return p if self._fallback else self.ir.predict(p)

print("✓ All utilities loaded")


## 3 · Dataset Loading & Summary

In [ ]:
datasets_dict = load_datasets()

rows = []
for name, d in datasets_dict.items():
    X, y = d['X'], d['y']
    counts = np.bincount(y)
    rows.append({
        'Dataset': name, 'Samples': len(X), 'Features': X.shape[1],
        'Classes': len(np.unique(y)),
        'Class names': ', '.join(d['target_names']),
        'Imbalance ratio': f"{counts.max()/counts.min():.2f}",
    })

summary = pd.DataFrame(rows).set_index('Dataset')
print("Dataset Summary")
print("=" * 65)
display(summary)


## 4 · Exploratory Data Analysis

In [ ]:
# Class distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Class Distributions', fontweight='bold')
colors = ['#185fa5', '#1d9e75', '#ba7517', '#a32d2d']

for ax, (name, d) in zip(axes, datasets_dict.items()):
    y, classes = d['y'], d['target_names']
    counts = [np.sum(y == i) for i in range(len(classes))]
    bars = ax.bar(classes, counts, color=colors[:len(classes)],
                  edgecolor='white', linewidth=1.5, width=0.5)
    ax.set_title(name.replace('_', ' ').title())
    ax.set_ylabel('Count')
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
                str(c), ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(counts) * 1.2)

plt.tight_layout()
plt.savefig('fig_class_distribution.png', bbox_inches='tight')
plt.show()
print("→ Saved fig_class_distribution.png")


In [ ]:
# Feature statistics summary
for name, d in datasets_dict.items():
    print(f"\n{'─'*50}  {name.upper()}")
    display(d['X'].describe().T[['mean','std','min','max']].head(8).round(3))
    print(f"  ... ({d['X'].shape[1]} features total)")


In [ ]:
# Correlation heatmap — breast_cancer
fig, ax = plt.subplots(figsize=(12, 9))
corr = datasets_dict['breast_cancer']['X'].iloc[:, :15].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.4, cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation — Breast Cancer (first 15 features)', pad=12)
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("→ Saved fig_correlation_heatmap.png")


## 5 · Build & Save Splits

In [ ]:
# Build 60/20/20 stratified splits and save to data/
splits = build_and_save_splits()

print()
print(f"{'Dataset':<20} {'Train':>8} {'Calib':>8} {'Test':>8}")
print('─' * 44)
for name, sp in splits.items():
    print(f"{name:<20} {len(sp['y_train']):>8} {len(sp['y_calib']):>8} {len(sp['y_test']):>8}")

print()
print("✓ Notebook 1 complete — run Notebook 2 next")
